# FlowR Seminar — Colab Workflow

Reproduziert die Vergleichsmethoden aus *FlowR: Flowing from Sparse to Dense 3D Reconstructions* (2504.01647) auf:

1. **eigenen Bildern** (Upload), und
2. einem **Paper-Beispieldatensatz** (Mip-NeRF 360 *garden*).

Schritte: GPU → Setup → Daten → Training pro Methode → Viewer per ngrok.

Konfiguration in `configs/nerfstudio.yaml`.

## 1. GPU & Runtime prüfen

In [ ]:
!nvidia-smi

## 2. Repo klonen

Ersetze `<USER>/<REPO>` durch deinen Fork.

In [ ]:
import os
REPO_URL = 'https://github.com/<USER>/<REPO>.git'
REPO_DIR = '/content/flowr-seminar'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 $REPO_URL $REPO_DIR
%cd $REPO_DIR

## 3. Setup (~10 min)

Installiert COLMAP, PyTorch+CUDA, tiny-cuda-nn, Nerfstudio, Nerfbusters, sowie klont alle Vergleichs-Repos in `external/`.

Schwergewichtige Methoden (FlowR, ViewCrafter, ZeroNVS, Zip-NeRF, GANeRF) werden **nur geklont**, nicht automatisch installiert — die Skripte und READMEs liegen unter `external/<name>/`.

In [ ]:
!bash scripts/setup_colab.sh

## 4. Datenquelle wählen

### 4a. Eigene Bilder hochladen

Lade 20–80 überlappende Fotos einer Szene hoch. Sie landen unter `data/custom/images/`.

In [ ]:
from google.colab import files
import os, shutil
os.makedirs('data/custom/images', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'data/custom/images/{name}')
print('Hochgeladen:', len(uploaded), 'Bilder')

### 4b. Paper-Beispiel (Mip-NeRF 360 *garden*)

In [ ]:
!ns-download-data mipnerf360 --capture-name=garden --save-dir data/demo

## 5. COLMAP-Posen für eigene Bilder (überspringen, wenn nur Demo trainiert wird)

In [ ]:
!ns-process-data images \
  --data data/custom/images \
  --output-dir data/custom/processed \
  --matching-method exhaustive

## 6. Methoden trainieren

Trainiert alle Nerfstudio-basierten Methoden aus `configs/nerfstudio.yaml` nacheinander — zuerst auf den eigenen Bildern, dann auf dem Demo-Set. Externe Methoden (FlowR, ViewCrafter, ...) werden übersprungen und manuell aus `external/` heraus gestartet.

In [ ]:
import yaml, subprocess, pathlib

cfg = yaml.safe_load(open('configs/nerfstudio.yaml'))
iters = cfg['train']['default_iterations']

datasets = {
    'custom': 'data/custom/processed',
    'garden': 'data/demo/mipnerf360/garden',
}

ns_methods = [m for m in cfg['methods'] if m['kind'].startswith('nerfstudio')]

for ds_name, ds_path in datasets.items():
    if not pathlib.Path(ds_path).exists():
        print(f'[skip] {ds_name} (Pfad fehlt: {ds_path})')
        continue
    for m in ns_methods:
        out = f'outputs/{ds_name}/{m["name"]}'
        print(f'\n=== {ds_name} :: {m["name"]} ===')
        subprocess.run([
            'bash', 'scripts/run_nerfstudio.sh',
            m['train_cmd'], ds_path, str(iters), out,
        ], check=False)

**Hinweis:** Der Block oben trainiert seriell und stoppt jedes Training nach `default_iterations` aus der YAML. Für einen schnellen Smoke-Test setze die Iterationen runter (z.B. 2000), bevor du `setup_colab.sh` erneut läufst.

## 7. Viewer-Tunnel

Statt einer Trainings-Schleife: ein einzelnes Training mit Live-Viewer. Schreibe deinen ngrok-Token in die Variable `NGROK_TOKEN` (oder vorher `os.environ['NGROK_AUTHTOKEN']` setzen).

In [ ]:
import os, subprocess, time
from scripts.tunnel import open_tunnel

NGROK_TOKEN = ''  # <- hier dein Token
METHOD = 'splatfacto'
DATA   = 'data/custom/processed'

proc = subprocess.Popen(
    ['bash', 'scripts/run_nerfstudio.sh', METHOD, DATA, '30000', f'outputs/viewer/{METHOD}'],
)
time.sleep(20)  # Nerfstudio braucht ein paar Sekunden bis der Websocket steht
viewer_url = open_tunnel(port=7007, token=NGROK_TOKEN or None)
print('Viewer:', viewer_url)

## 8. Aufräumen

Resultate liegen in `outputs/<dataset>/<method>/`. Per Drag-and-Drop in die Colab-Dateiansicht herunterladbar, oder per `!zip` und `files.download`.

In [ ]:
# Beispiel: alle Custom-Resultate zippen und herunterladen
# !zip -r outputs_custom.zip outputs/custom
# from google.colab import files; files.download('outputs_custom.zip')